In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pymargins import Margins, SurveyDesign

# Load the stratified sample generated from R's survey package
apistrat = pd.read_csv("data/apistrat.csv")
print(apistrat.shape)
print(apistrat[["stype", "api00", "api99", "meals", "ell", "avg.ed",
                "mobility", "pw", "fpc"]].head())

(200, 39)
  stype  api00  api99  meals  ell  avg.ed  mobility         pw   fpc
0     E    840    816     33   25    3.32        11  44.209999  4421
1     E    516    476     98   77    1.67        26  44.209999  4421
2     E    531    544     64   23    2.34        17  44.209999  4421
3     E    501    457     83   63    1.86        13  44.209999  4421
4     E    720    659     26   17    3.17        31  44.209999  4421


In [2]:
print("Schools per stratum:")
print(apistrat["stype"].value_counts().sort_index())
print("\nWeight range: {:.1f} – {:.1f}".format(apistrat["pw"].min(),
                                                apistrat["pw"].max()))
print("FPC range:    {:.1f} – {:.1f}".format(apistrat["fpc"].min(),
                                                apistrat["fpc"].max()))

Schools per stratum:
stype
E    100
H     50
M     50
Name: count, dtype: int64

Weight range: 15.1 – 44.2
FPC range:    755.0 – 4421.0


In [3]:
fit = smf.glm(
    "api00 ~ meals + ell + Q(\"avg.ed\") + mobility",
    data=apistrat,
    freq_weights=apistrat["pw"].values,
).fit()
print(fit.summary().tables[1])

                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept     532.8152      8.719     61.108      0.000     515.726     549.905
meals          -1.8421      0.058    -31.534      0.000      -1.957      -1.728
ell            -0.0022      0.062     -0.036      0.972      -0.123       0.119
Q("avg.ed")    77.3537      2.285     33.851      0.000      72.875      81.832
mobility        0.1594      0.076      2.110      0.035       0.011       0.307


In [4]:
# stype is 'E', 'H', 'M' — encode as integers for SurveyDesign
stratum_codes = apistrat["stype"].astype("category").cat.codes

survey = SurveyDesign(
    weights=apistrat["pw"].values,
    strata=stratum_codes.values,
)

m = Margins(fit, survey_design=survey,
            weights=apistrat["pw"].values, at="overall")
print(m.dydx("meals").summary())

            Margins Result (delta, level=0.95)            
       estimate  std err        z  P>|z|  [95% Conf. Int.]
----------------------------------------------------------
meals   -1.8421   0.4204  -4.3820  0.000  -2.6661, -1.0182

n = 200
κ: 0.000
Delta-vs-sim disagreement: 2.130%


In [5]:
m_boot = Margins(
    fit,
    survey_design=survey,
    weights=apistrat["pw"].values,
    at="overall",
    method="bootstrap",
    n_boot=500,
    rng_seed=42,
)
print(m_boot.dydx("meals").summary())

           Margins Result (bootstrap, level=0.95)           
       estimate  std err  statistic  P>|z|  [95% Conf. Int.]
------------------------------------------------------------
meals   -1.8421   0.4409    -1.8421  0.008  -2.1456, -0.4205

n = 200
κ: 0.000


In [6]:
results = pd.DataFrame({
    "approach": ["survey linearization", "survey bootstrap"],
    "estimate": [
        float(m.dydx("meals").estimate),
        float(m_boot.dydx("meals").estimate),
    ],
    "std_error": [
        float(m.dydx("meals").std_error),
        float(m_boot.dydx("meals").std_error),
    ],
})
print(results.round(4))

               approach  estimate  std_error
0  survey linearization   -1.8421     0.4204
1      survey bootstrap   -1.8421     0.4409


In [7]:
from pymargins import pairwise

scen, w = pairwise("ell", [5, 35])
res = m.contrasts(scenarios=scen, contrasts=w)
print(res.summary())

            Margins Result (delta, level=0.95)            
       estimate  std err       z  P>|z|   [95% Conf. Int.]
----------------------------------------------------------
ell=5    0.0656  11.9263  0.0055  0.996  -23.3094, 23.4407

n = 200
κ: 0.000
Delta-vs-sim disagreement: 1330.407%
